# GeoSPARQL Extension Functions (rdflib-geosparql)

The following functions are implemented in rdflib-geosparql, but are not (yet) officially standardized by OGC.


## Installing requirements

In [1]:
!pip install -r requirements.txt --break-system-packages
!pip install ipyleaflet pyvista --break-system-packages
!pwd
!ls

import rdflib
import json
import pyvista as pv
import trimesh
import shapely.geometry
from shapely import wkt
from shapely.ops import unary_union
from io import BytesIO
from shapely.ops import transform
from rdflib import Graph, Literal
import os
import itertools
from shapely.testing import assert_geometries_equal
from geosparql.geosparql import LiteralUtils
from geosparql.geosparql_aggregates import processLiteralTypeToGeom
from IPython.display import display, HTML
import ipywidgets as W
from ipyleaflet import Map, WKTLayer, GeoJSON, FullScreenControl, LayersControl, Popup#, Tooltip
from geosparql.geosparql import geosparql13

print("Total: "+str(len(geosparql13))+" functions")


mapcenter=(34.1,-83.2)
zoomlevel=10
GEO = "http://www.opengis.net/ont/geosparql#"
GEOF = "http://www.opengis.net/def/function/geosparql/"
GEOFEXT = "http://www.opengis.net/def/function/geosparql/ext/"

def getHTMLStringFromQueryResult(qres):
    res="<table><tr><th>Variable</th><th>Value</th></tr>"
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
        for row in resultlist:
            for item in row:
                if isinstance(row[item],Literal) and row[item].datatype!=None:
                    res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item].datatype)+"\">"+str(row[item]).strip().replace("<","&lt;").replace(">","&gt;").replace("\n","<br/>")+"</a></td></tr>"
                elif isinstance(row[item],URIRef):
                    res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item])+"\">"+str(row[item]).strip()+"</a></td></tr>"
                else:
                    res+="<tr><td>"+str(item)+"</td><td>"+str(row[item]).strip().replace("<","&lt;").replace(">","&gt;").replace("\n","<br/>")+"</td></tr>"
    res+="</table>"
    return res

def getMapFromWKTResult(qres,rows=[],lmap=None):
    layers=[]
    lastcentroid=shapely.geometry.Point(mapcenter[0],mapcenter[1])
    bboxes=[]
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
        for row in rows:
            if resultlist!=None and row in resultlist[0]:
                popup="<h3>"+str(row)+"</h3><ul>"        
                for rowres in resultlist:
                    for item in rowres:
                        if isinstance(rowres[item],Literal) and rowres[item].datatype!=None:
                            popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item].datatype)+"\">"+str(rowres[item]).strip().replace("<","&lt;").replace(">","&gt;").replace("\n","<br/>")+"</a></li>"
                        elif isinstance(rowres[item],URIRef):
                            popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item])+"\">"+str(rowres[item]).strip()+"</a></li>"
                        else:
                            popup+="<li><b>"+str(item)+":</b> "+str(rowres[item]).strip().replace("<","&lt;").replace(">","&gt;").replace("\n","<br/>")+"</li>"
                popup+="</ul>"
                toprocess=resultlist[0][row].strip()
                if toprocess.startswith("<http"):
                    toprocess=toprocess[toprocess.find(">")+1:]
                geom = wkt.loads(toprocess)
                if not shapely.is_empty(geom):
                    lastcentroid=geom.centroid
                    nlayer=WKTLayer(name=str(row),wkt_string=geom.wkt,hover_style={"fillColor": "red"})
                    bounds=geom.bounds
                    bboxes.append(shapely.geometry.box(bounds[0],bounds[1],bounds[2],bounds[3]))
                    msg=W.HTML()
                    msg.value=popup
                    nlpopup=Popup(
                        location=[lastcentroid.y,lastcentroid.x],
                        child=msg,
                        close_button=True,
                        auto_close=False,
                        close_on_escape_key=False
                    )
                    nlayer.popup=msg
                    layers.append(nlayer)
    if lmap==None:
        maxbounds=unary_union(bboxes).bounds
        lmap=Map(center=(lastcentroid.y,lastcentroid.x), zoom=zoomlevel)
        control = FullScreenControl()
        lmap.add(control)
        control = LayersControl(position='topright')
        lmap.add(control)
    for lay in layers:
        lmap.add(lay)
    lmap.fit_bounds(((maxbounds[1],maxbounds[2]),(maxbounds[3],maxbounds[0])))
    return lmap    

def processQueryResults(qres,rows=[],lmap=None,rows3d=[]):
    display(HTML(getHTMLStringFromQueryResult(qres)))
    if rows!=[]:
        lmap=getMapFromWKTResult(qres,rows,lmap)
    if rows3d!=[]:
        if qres is not None and len(qres.bindings)>0:
            resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
            for rw in rows3d:
                for res in resultlist:
                    if rw in res:
                        tm = trimesh.load(BytesIO(str(res[rw]).encode()), file_type=rw)
                        # Convert Trimesh -> PyVista
                        mesh = pv.wrap(tm)        
                        mesh.plot(notebook=True)
                        tm.show()
                        #mesh = pv.read("model.ply")
                        #plotter = pv.Plotter(notebook=True)
                        #plotter.add_mesh(mesh)
                        #plotter.show()
    return lmap

g=Graph()
g.parse("tests/testdata.ttl")
print(len(g))

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
/home/timo/git/rdflib-geosparql
docs		   GeoSPARQLExt.ipynb  rdflib-geosparql        setup.py
geosparql	   LICENSE	       rdflib-geosparql.ipynb  tests
GeoSPARQL10.ipynb  pyproject.toml      README.md
GeoSPARQL11.ipynb  pytest.ini	       requirements.txt

Registered custom function http://www.opengis.net/def/function/geosparql/boundary
Registered custom function http://www.opengis.net/def/function/geosparql/buffer
Registered custom function http://www.opengis.net/def/function/geosparql/convexHull
Registered custom function http://www.opengis.net/def/function/geosparql/difference
Registered custom function http://www.opengis.net/def/function/geosparql/distance
Registered custom function http://www.opengis.net/def/function/geosparql/ehContains
Registered custom function http://www.opengis.net/def/function/geosparql/ehCoveredBy
Registe

## Geometry Serializations

Functions in this section allow for the conversion of GeoSPARQL geometries to other literal formats.

### geofext:asGeoCode Function

Converts a given geometry to a geocode representation as [geo:geoCodeLiteral](http://www.opengis.net/ont/geosparql#geoCodeLiteral)

In [2]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gc ?asGeoURI ?asGeoHash
WHERE {
  my:A geo:hasGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGeocode(?aLiteral,"http://opengis.net/ont/geocode/OpenLocationCode") as ?gc)
  BIND (geof:asGeoCode(?aLiteral,"http://opengis.net/ont/geocode/GeoURI") as ?asGeoURI)
  BIND (geof:asGeoCode(?aLiteral,"http://opengis.net/ont/geocode/GeoHash") as ?asGeoHash)
}
"""),["aLiteral"],None)

POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
http://opengis.net/ont/geocode/OpenLocationCode


Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gc,<http://opengis.net/ont/geocode/OpenLocationCode> 2G8PJ822+22


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asGeoYAML Function

Returns a GeoYAML representation of a given geometry typed as [geo:geoYAMLLiteral](http://www.opengis.net/ont/geosparql#geoYAMLLiteral)

In [3]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?geoyaml
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGeoYAML(?aLiteral) as ?geoyaml)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
geoyaml,coordinates:- - - -83.6 - 34.1 - - -83.2 - 34.1 - - -83.2 - 34.5 - - -83.6 - 34.5 - - -83.6 - 34.1type: Polygon


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asGLTF Function

In [4]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gltf
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGLTF(?aLiteral) as ?gltf)
}
"""),["aLiteral"],None)

{'gltf_buffer_0.bin': b'\x01\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x03\x00\x00\x00\x02\x00\x00\x00\x01\x00\x00\x00\x07\x00\x00\x00\x04\x00\x00\x00\x05\x00\x00\x00\x05\x00\x00\x00\x06\x00\x00\x00\x07\x00\x00\x00\x05\x00\x00\x00\x04\x00\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x04\x00\x00\x00\x00\x00\x00\x00\x06\x00\x00\x00\x05\x00\x00\x00\x02\x00\x00\x00\x02\x00\x00\x00\x05\x00\x00\x00\x01\x00\x00\x00\x04\x00\x00\x00\x07\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x07\x00\x00\x00\x03\x00\x00\x00\x07\x00\x00\x00\x06\x00\x00\x00\x03\x00\x00\x00\x03\x00\x00\x00\x06\x00\x00\x00\x02\x00\x00\x00', 'gltf_buffer_1.bin': b'33\xa7\xc2ff\x08B\x00\x00\x00\x00ff\xa6\xc2ff\x08B\x00\x00\x00\x00ff\xa6\xc2\x00\x00\nB\x00\x00\x00\x0033\xa7\xc2\x00\x00\nB\x00\x00\x00\x0033\xa7\xc2ff\x08B\xcd\xcc\xcc=ff\xa6\xc2ff\x08B\xcd\xcc\xcc=ff\xa6\xc2\x00\x00\nB\xcd\xcc\xcc=33\xa7\xc2\x00\x00\nB\xcd\xcc\xcc=', 'model.gltf': b'{"scene":0,"scenes":[{"nodes":[0]}],"asset":{"version":"2.0","generator":"https://githu

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gltf,"{'gltf_buffer_0.bin': b'\x01\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x03\x00\x00\x00\x02\x00\x00\x00\x01\x00\x00\x00\x07\x00\x00\x00\x04\x00\x00\x00\x05\x00\x00\x00\x05\x00\x00\x00\x06\x00\x00\x00\x07\x00\x00\x00\x05\x00\x00\x00\x04\x00\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x04\x00\x00\x00\x00\x00\x00\x00\x06\x00\x00\x00\x05\x00\x00\x00\x02\x00\x00\x00\x02\x00\x00\x00\x05\x00\x00\x00\x01\x00\x00\x00\x04\x00\x00\x00\x07\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x07\x00\x00\x00\x03\x00\x00\x00\x07\x00\x00\x00\x06\x00\x00\x00\x03\x00\x00\x00\x03\x00\x00\x00\x06\x00\x00\x00\x02\x00\x00\x00', 'gltf_buffer_1.bin': b'33\xa7\xc2ff\x08B\x00\x00\x00\x00ff\xa6\xc2ff\x08B\x00\x00\x00\x00ff\xa6\xc2\x00\x00\nB\x00\x00\x00\x0033\xa7\xc2\x00\x00\nB\x00\x00\x00\x0033\xa7\xc2ff\x08B\xcd\xcc\xcc=ff\xa6\xc2ff\x08B\xcd\xcc\xcc=ff\xa6\xc2\x00\x00\nB\xcd\xcc\xcc=33\xa7\xc2\x00\x00\nB\xcd\xcc\xcc=', 'model.gltf': b'{""scene"":0,""scenes"":[{""nodes"":[0]}],""asset"":{""version"":""2.0"",""generator"":""https://github.com/mikedh/trimesh""},""accessors"":[{""componentType"":5125,""type"":""SCALAR"",""bufferView"":0,""count"":36,""max"":[7],""min"":[0]},{""componentType"":5126,""type"":""VEC3"",""byteOffset"":0,""bufferView"":1,""count"":8,""max"":[-83.19999694824219,34.5,0.10000000149011612],""min"":[-83.5999984741211,34.099998474121094,0.0]}],""meshes"":[{""name"":""geometry_0"",""extras"":{""processed"":true},""primitives"":[{""attributes"":{""POSITION"":1},""indices"":0,""mode"":4}]}],""nodes"":[{""name"":""world"",""children"":[1]},{""name"":""geometry_0"",""mesh"":0}],""buffers"":[{""uri"":""gltf_buffer_0.bin"",""byteLength"":144},{""uri"":""gltf_buffer_1.bin"",""byteLength"":96}],""bufferViews"":[{""buffer"":0,""byteOffset"":0,""byteLength"":144},{""buffer"":1,""byteOffset"":0,""byteLength"":96}]}'}"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asGPX Function

In [5]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gpx
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGPX(?aLiteral) as ?gpx)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gpx,"<?xml version=""1.0"" encoding=""UTF-8""?><gpx xmlns=""http://www.topografix.com/GPX/1/1"" xmlns:xsi=""http://www.w3.org/2001/XMLSchema-instance"" xsi:schemaLocation=""http://www.topografix.com/GPX/1/1 http://www.topografix.com/GPX/1/1/gpx.xsd"" version=""1.1"" creator=""gpx.py -- https://github.com/tkrajina/gpxpy""> <trk> <trkseg> <trkpt lat=""34.1"" lon=""-83.6""> </trkpt> <trkpt lat=""34.1"" lon=""-83.2""> </trkpt> <trkpt lat=""34.5"" lon=""-83.2""> </trkpt> <trkpt lat=""34.5"" lon=""-83.6""> </trkpt> <trkpt lat=""34.1"" lon=""-83.6""> </trkpt> </trkseg> </trk></gpx>"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asJSONFG Function

Returns a JSONFG version of a given geometry typed as [geo:jsonfgLiteral](http://www.opengis.net/ont/geosparql#jsonfgLiteral)

In [6]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?jsonfg
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asJSONFG(?aLiteral) as ?jsonfg)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
jsonfg,"{""type"": ""Polygon"", ""coordinates"": [[[-83.6, 34.1], [-83.2, 34.1], [-83.2, 34.5], [-83.6, 34.5], [-83.6, 34.1]]], ""coordRefSys"": ""http://www.opengis.net/def/crs/OGC/1.3/CRS84"", ""conformsTo"": [""http://www.opengis.net/spec/json-fg-1/1.0/conf/core"", ""http://www.opengis.net/spec/json-fg-1/1.0/conf/types-schemas""]}"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asOBJ Function

In [7]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?obj
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asOBJ(?aLiteral) as ?obj)
}
"""),["aLiteral"],None,["obj"])

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asPLY Function

In [8]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?ply
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asPLY(?aLiteral) as ?ply)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
ply,plyformat ascii 1.0comment https://github.com/mikedh/trimeshelement vertex 8property float xproperty float yproperty float zelement face 12property list uchar int vertex_indicesend_header-83.59999847 34.09999847 0.00000000-83.19999695 34.09999847 0.00000000-83.19999695 34.50000000 0.00000000-83.59999847 34.50000000 0.00000000-83.59999847 34.09999847 0.10000000-83.19999695 34.09999847 0.10000000-83.19999695 34.50000000 0.10000000-83.59999847 34.50000000 0.100000003 1 0 33 3 2 13 7 4 53 5 6 73 5 4 13 1 4 03 6 5 23 2 5 13 4 7 03 0 7 33 7 6 33 3 6 2


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asSVG Function

Returns an SVG path representation of the outline of the given geometry as [geo:svgLiteral](http://www.opengis.net/ont/geosparql#svgLiteral)

In [9]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?svg
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asSVG(?aLiteral) as ?svg)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
svg,"<path fill-rule=""evenodd"" fill=""#66cc99"" stroke=""#555555"" stroke-width=""2.0"" opacity=""0.6"" d=""M -83.6,34.1 L -83.2,34.1 L -83.2,34.5 L -83.6,34.5 L -83.6,34.1 z"" />"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:asWKB Function

Converts a given geometry to its Well-Known Binary representation typed as [geo:wkbLiteral](http://www.opengis.net/ont/geosparql#wkbLiteral)

In [10]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?wkb
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asWKB(?aLiteral) as ?wkb)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
wkb,010300000001000000050000006666666666E654C0CDCCCCCCCC0C4140CDCCCCCCCCCC54C0CDCCCCCCCC0C4140CDCCCCCCCCCC54C000000000004041406666666666E654C000000000004041406666666666E654C0CDCCCCCCCC0C4140


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## Directional relation functions

This section introduces directional relation functions in rdflib-geosparql.

### geofext:above Function ⬜️🧊

Checks if the first input geometry is above the second input geometry and returns the result as [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean). Geometry coordinate reference systems are normalized if they differ.

In [11]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?abovee) as ?D_above_A) (xsd:boolean(?nabovee) as ?A_above_D) (xsd:boolean(?abovee3d) as ?D_above_A_3D) (xsd:boolean(?nabovee3d) as ?A_above_D_3D)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND("Polygon Z ((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"^^geo:wktLiteral AS ?dLiteral3d)
  BIND("Polygon Z ((-83.6 34.1 10.0, -83.2 34.1 4.5, -83.2 34.5 4.5, -83.6 34.5 10.0, -83.6 34.1 10.0))"^^geo:wktLiteral AS ?aLiteral3d)
  BIND (geof:above(?aLiteral, ?dLiteral) as ?nabovee)
  BIND (geof:above(?dLiteral, ?aLiteral) as ?abovee)
  BIND (geof:above(?aLiteral3d, ?dLiteral3d) as ?nabovee3d)
  BIND (geof:above(?dLiteral3d, ?aLiteral3d) as ?abovee3d)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"
D_above_A,false
A_above_D,true
D_above_A_3D,false
A_above_D_3D,true


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:behind Function

Checks if the first input geometry is behind the second input geometry and returns the result as [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean). Geometry coordinate reference systems are normalized if they differ.

In [12]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?beloww) as ?D_Behind_A) (xsd:boolean(?nbeloww) as ?A_Behind_D)
WHERE {
  BIND("Polygon Z((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"^^geo:wktLiteral AS ?aLiteral)
  BIND("Polygon Z((-83.3 33.0 10.0, -83.1 33.0 10.0, -83.1 33.2 10.0, -83.3 33.2 10.0, -83.3 33.0 10.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:behind(?aLiteral, ?dLiteral) as ?nbeloww)
  BIND (geof:behind(?dLiteral, ?aLiteral) as ?beloww)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon Z((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"
dLiteral,"Polygon Z((-83.3 33.0 10.0, -83.1 33.0 10.0, -83.1 33.2 10.0, -83.3 33.2 10.0, -83.3 33.0 10.0))"
D_Behind_A,false
A_Behind_D,true


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:below Function

Checks if the first input geometry is below the second input geometry and returns the result as [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean). Geometry coordinate reference systems are normalized if they differ.

In [13]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?beloww) as ?D_Below_A) (xsd:boolean(?nbeloww) as ?A_Below_D) (xsd:boolean(?beloww3d) as ?D_below_A_3D) (xsd:boolean(?nbeloww3d) as ?A_below_D_3D)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND("Polygon Z ((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"^^geo:wktLiteral AS ?dLiteral3d)
  BIND("Polygon Z ((-83.6 34.1 10.0, -83.2 34.1 4.5, -83.2 34.5 4.5, -83.6 34.5 10.0, -83.6 34.1 10.0))"^^geo:wktLiteral AS ?aLiteral3d)
  BIND (geof:below(?aLiteral, ?dLiteral) as ?nbeloww)
  BIND (geof:below(?dLiteral, ?aLiteral) as ?beloww)
  BIND (geof:below(?aLiteral3d, ?dLiteral3d) as ?nbeloww3d)
  BIND (geof:below(?dLiteral3d, ?aLiteral3d) as ?beloww3d)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-83.3 33.0, -83.1 33.0, -83.1 33.2, -83.3 33.2, -83.3 33.0))"
D_Below_A,true
A_Below_D,false
D_below_A_3D,true
A_below_D_3D,false


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:inFrontOf Function

Checks if the first input geometry is in front of the second input geometry and returns the result as [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean). Geometry coordinate reference systems are normalized if they differ.

In [14]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?beloww) as ?D_InFrontOf_A) (xsd:boolean(?nbeloww) as ?A_InFrontOf_D)
WHERE {
  BIND("Polygon Z((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"^^geo:wktLiteral AS ?aLiteral)
  BIND("Polygon Z((-83.3 33.0 10.0, -83.1 33.0 10.0, -83.1 33.2 10.0, -83.3 33.2 10.0, -83.3 33.0 10.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND (geof:inFrontOf(?aLiteral, ?dLiteral) as ?nbeloww)
  BIND (geof:inFrontOf(?dLiteral, ?aLiteral) as ?beloww)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"Polygon Z((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"
dLiteral,"Polygon Z((-83.3 33.0 10.0, -83.1 33.0 10.0, -83.1 33.2 10.0, -83.3 33.2 10.0, -83.3 33.0 10.0))"
D_InFrontOf_A,true
A_InFrontOf_D,false


Map(center=[33.1, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:leftOf Function

Checks if the first input geometry is left of the second input geometry and returns the result as [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean). Geometry coordinate reference systems are normalized if they differ.

In [15]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?leftt) as ?A_leftOf_D) (xsd:boolean(?nleftt) as ?D_LeftOf_A) (xsd:boolean(?leftt3d) as ?D_leftOf_A_3D) (xsd:boolean(?nleftt3d) as ?A_leftOf_D_3D)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND("Polygon Z ((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"^^geo:wktLiteral AS ?dLiteral3d)
  BIND("Polygon Z ((-83.6 34.1 10.0, -83.2 34.1 4.5, -83.2 34.5 4.5, -83.6 34.5 10.0, -83.6 34.1 10.0))"^^geo:wktLiteral AS ?aLiteral3d)
  BIND (geof:leftOf(?aLiteral, ?dLiteral) as ?leftt)
  BIND (geof:leftOf(?dLiteral, ?aLiteral) as ?nleftt)
  BIND (geof:leftOf(?aLiteral3d, ?dLiteral3d) as ?nleftt3d)
  BIND (geof:leftOf(?dLiteral3d, ?aLiteral3d) as ?leftt3d)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"
A_leftOf_D,true
D_LeftOf_A,false
D_leftOf_A_3D,false
A_leftOf_D_3D,false


Map(center=[34.099999999999994, -82.2], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_ti…

### geofext:rightOf Function

Checks if the first input geometry is right of the second input geometry and returns the result as [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean). Geometry coordinate reference systems are normalized if they differ.

In [16]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral (xsd:boolean(?rightt) as ?A_rightOf_D) (xsd:boolean(?nrightt) as ?D_rightOf_A) (xsd:boolean(?rightt3d) as ?D_rightOf_A_3D) (xsd:boolean(?nrightt3d) as ?A_rightOf_D_3D)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND("Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"^^geo:wktLiteral AS ?dLiteral)
  BIND("Polygon Z ((-83.3 33.0 5.0, -83.1 33.0 5.0, -83.1 33.2 5.0, -83.3 33.2 5.0, -83.3 33.0 5.0))"^^geo:wktLiteral AS ?dLiteral3d)
  BIND("Polygon Z ((-83.6 34.1 10.0, -83.2 34.1 4.5, -83.2 34.5 4.5, -83.6 34.5 10.0, -83.6 34.1 10.0))"^^geo:wktLiteral AS ?aLiteral3d)
  BIND (geof:rightOf(?aLiteral, ?dLiteral) as ?nrightt)
  BIND (geof:rightOf(?dLiteral, ?aLiteral) as ?rightt)
  BIND (geof:rightOf(?aLiteral3d, ?dLiteral3d) as ?nrightt3d)
  BIND (geof:rightOf(?dLiteral3d, ?aLiteral3d) as ?rightt3d)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"Polygon((-82.3 34.0, -82.1 34.0, -82.1 34.2, -82.3 34.2, -82.3 34.0))"
A_rightOf_D,true
D_rightOf_A,false
D_rightOf_A_3D,false
A_rightOf_D_3D,false


Map(center=[34.099999999999994, -82.2], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_ti…

## Geometry Accessor Functions

These functions expose attributes of geometries and calculate geometry specific derivations

### geofext:boundingDiagonal Function

Returns the diagonal of a geometry's bounding box, i.e. a 2 point LineString from its minimum to maximum coordinates.

In [17]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bdiag
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:boundingDiagonal(?aLiteral) as ?bdiag)
}
"""),["aLiteral","bdiag"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bdiag,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LINESTRING (-83.6 34.1, -83.2 34.5)"


Map(center=[34.3, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:compactnessRatio Function

Calculates the compactness ratio of a polygon, an indicator of polygon shape complexity and returns is as [xsd:double](http://www.w3.org/2001/XMLSchema#double)

In [18]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?cratio
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:compactnessRatio(?aLiteral) as ?cratio)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
cratio,0.886226925452758


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:endPoint Function ⬜️🧊

Returns the last point of a given geometry in the CRS and literal format of the input geometry.

In [19]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?endPoint
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:endPoint(?literal) AS ?endPoint)
}
"""),["literal","endPoint"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
endPoint,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT (-83.6 34.1)


Map(center=[34.1, -83.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:exteriorRing Function

Returns the exterior ring of a given polygon in the CRS and literal format of the input geometry

In [20]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?exRing
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:exteriorRing(?literal) AS ?exRing)
}
"""),["literal","exRing"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
exRing,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LINEARRING (-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1)"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:geometricMedian Function

Calculates the geometric median of a geometry and returns a Point geometry in the literal format and CRS of the input geometry.

In [21]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?gmedian {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:geometricMedian(?a_wkt) AS ?gmedian)
}
"""),["a_wkt","gmedian"],None)

Variable,Value
a_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
gmedian,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT (-83.515470050852 34.184529949148)


Map(center=[34.184529949148, -83.515470050852], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zo…

### geofext:isCCW Function

In [22]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX geofo: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?e_wkt ?isACW ?isECW {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isCCW(?a_wkt) AS ?isACW)
    <http://example.org/ApplicationSchema#EExactGeom> geo:asWKT ?e_wkt .
    BIND(geof:isCCW(?e_wkt) AS ?isECW)
}
"""),["a_wkt","e_wkt"],None)

Variable,Value
a_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
e_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LineString(-83.4 34.0, -83.3 34.3)"


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:isCW Function

In [23]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX geofo: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?e_wkt ?isACW ?isECW {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isCW(?a_wkt) AS ?isACW)
    <http://example.org/ApplicationSchema#EExactGeom> geo:asWKT ?e_wkt .
    BIND(geof:isCW(?e_wkt) AS ?isECW)
}
"""),["a_wkt","e_wkt"],None)

Variable,Value
a_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
e_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LineString(-83.4 34.0, -83.3 34.3)"


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:isClosed Function

Tests if a geometry's start and endpoint are equal and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [24]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?e_wkt ?isAClosed ?isEClosed {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isClosed(?a_wkt) AS ?isAClosed)
    <http://example.org/ApplicationSchema#EExactGeom> geo:asWKT ?e_wkt .
    BIND(geof:isClosed(?e_wkt) AS ?isEClosed)
}
"""),["a_wkt","e_wkt"],None)

Variable,Value
a_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isAClosed,true
e_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LineString(-83.4 34.0, -83.3 34.3)"
isEClosed,false


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:isCollection Function

Tests if a geometry is of a geometry collection type including a MUTLI type and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [25]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?C1 ?C2 ?C3 ?isC1 ?isC2 ?isC3 {
    BIND("GEOMETRYCOLLECTION (MULTIPOINT((0 0), (1 1)), POINT(3 4), LINESTRING(2 3, 3 4))"^^geo:wktLiteral as ?C1)
    BIND("MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))"^^geo:wktLiteral as ?C2)
    BIND("POLYGON ((0 0, 4 0, 4 4, 0 4, 0 0), (1 1, 1 2, 2 2, 2 1, 1 1))"^^geo:wktLiteral as ?C3)
    BIND(geof:isCollection(?C1) AS ?isC1)
    BIND(geof:isCollection(?C2) AS ?isC2)
    BIND(geof:isCollection(?C3) AS ?isC3)
}
"""),["C1","C2","C3"],None)

Variable,Value
C1,"GEOMETRYCOLLECTION (MULTIPOINT((0 0), (1 1)), POINT(3 4), LINESTRING(2 3, 3 4))"
C2,"MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))"
C3,"POLYGON ((0 0, 4 0, 4 4, 0 4, 0 0), (1 1, 1 2, 2 2, 2 1, 1 1))"
isC1,true
isC2,true
isC3,false


Map(center=[2.033333333333333, 2.033333333333333], controls=(ZoomControl(options=['position', 'zoom_in_text', …

### geofext:isRectangle Function

Checks if a given geometry is rectangular and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [26]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX geofo: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?isRectangle ?isNoRectangle {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isRectangle(?a_wkt) AS ?isNoRectangle)
    BIND(geof:isRectangle(geofo:envelope(?a_wkt)) AS ?isRectangle)
}
"""),["a_wkt"],None)

Variable,Value
a_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isNoRectangle,true
isRectangle,true


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:isRing Function

Checks if a geometry is simple and closed and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [27]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT  ?a_wkt ?o_wkt ?isARing ?isORing {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isRing(?a_wkt) AS ?isARing)
            <http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
    BIND(geof:isRing("POINT M (1 2 3)"^^geo:wktLiteral) AS ?isORing)
}
"""),["a_wkt","o_wkt"],None)

Variable,Value
a_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isARing,true
o_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
isORing,false


Map(center=[47.47714370265989, 2.4756124648711317], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### geofext:isTriangle Function

Checks whether a given geometry is a triangle and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [28]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?isTriangle ?isNoTriangle {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isTriangle(?a_wkt) AS ?isNoTriangle)
    BIND(geof:isTriangle("POLYGON ((0 0,0 1,1 1,0 0))"^^geo:wktLiteral) AS ?isTriangle)
}
"""),["a_wkt"],None)

Variable,Value
a_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isNoTriangle,false
isTriangle,true


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:isValid 

Checks whether a given geometry is a valid and returns an [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [29]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?o_wkt ?a_isValid ?o_isNotValid {
<http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
BIND(geof:isValid(?a_wkt) AS ?a_isValid)
        <http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
BIND(geof:isValid("POLYGON((0 0, 10 10, 0 10, 10 0, 0 0))"^^geo:wktLiteral) AS ?o_isNotValid)
}
"""),["a_wkt","o_wkt"],None)

Variable,Value
a_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
a_isValid,true
o_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
o_isNotValid,false


Map(center=[47.47714370265989, 2.4756124648711317], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### geofext:isValidTrajectory Function

In [30]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?o_wkt ?validTGeom ?isValidT ?O_isValidT2 ?A_isValidT {
    <http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
    BIND(geof:isValidTrajectory(?a_wkt) AS ?A_isValidT)
    <http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
    BIND(geof:isValidTrajectory(?o_wkt) AS ?O_isValidT2)
    BIND("LineString M(0 0 1, 10 10 2, 0 10 3, 10 0 4, 0 0 5)"^^geo:wktLiteral AS ?validTGeom)
    BIND(geof:isValidTrajectory(?validTGeom) AS ?isValidT)
}
"""),["a_wkt","o_wkt","validTGeom"],None)

Variable,Value
a_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
A_isValidT,false
o_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
O_isValidT2,false
validTGeom,"LineString M(0 0 1, 10 10 2, 0 10 3, 10 0 4, 0 0 5)"
isValidT,true


Map(center=[5.0, 5.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

### geofext:M Function

Returns the measurement coordinate of a point geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double).

In [31]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?m
WHERE {
  my:PPointGeom geo:asWKT ?literal .
  BIND(geof:M(?literal) AS ?m)
}
"""),["literal"],None)

Variable,Value
literal,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point M (-83.4 34.0 5.0)
m,5.0


Map(center=[34.0, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:maxM Function

Returns the maximum measurement coordinate of a given geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double)

In [32]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?maxM
WHERE {
  my:PExactGeom geo:asWKT ?literal .
  BIND(geof:maxM(?literal) AS ?maxM)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LineString M (-83.4 34.0 5, -83.3 34.3 10)"
maxM,10.0


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:minM Function

Returns the minimum measurement coordinate of a geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double)

In [33]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?minM
WHERE {
  my:PExactGeom geo:asWKT ?literal .
  BIND(geof:minM(?literal) AS ?minM)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LineString M (-83.4 34.0 5, -83.3 34.3 10)"
minM,5.0


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:numInteriorRing Function ⬜️🧊

Returns the number of interior rings of a geometry as [xsd:integer](http://www.w3.org/2001/XMLSchema#integer)

In [34]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?numInteriorRing
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:numInteriorRing(?literal) AS ?numInteriorRing)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
numInteriorRing,0


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:numPatches Function

In [35]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?numPatches
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:numPatches(?literal) AS ?numPatches)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
numPatches,1


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:numPoints Function ⬜️🧊

Returns the number of points in a given geometry as [xsd:integer](http://www.w3.org/2001/XMLSchema#integer).

In [36]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?numPoints
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:numPoints(?literal) AS ?numPoints)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
numPoints,5


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:pointN Function ⬜️🧊

Returns the nth point of a given geometry in the literal format and CRS of the input geometry.

In [37]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?pointN
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:pointN(?literal,"1"^^xsd:integer) AS ?pointN)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
pointN,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT (-83.2 34.1)


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:selfIntersections Function

Calculates selfintersection points and returns them as a MultiPoint Geometry in the literal format and CRS of the input geometry

In [38]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?selfIPoints
WHERE {
  my:OExactGeom geo:asWKT ?literal .
  BIND(geof:selfIntersections(?literal) AS ?selfIPoints)
}
"""),["literal","selfIPoints"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
selfIPoints,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> MULTIPOINT ((2.47217 47.52403), (2.4332749328587093 47.45651545608854), (2.4211170488384783 47.475258601765105))"


Map(center=[47.48526801928455, 2.442187327232396], controls=(ZoomControl(options=['position', 'zoom_in_text', …

### geofext:startPoint Function ⬜️🧊

Returns the first point of a given geometry in the CRS and literal format of the input geometry.

In [39]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?startPoint
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:startPoint(?literal) AS ?startPoint)
}
"""),["literal","startPoint"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
startPoint,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT (-83.6 34.1)


Map(center=[34.1, -83.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:X Function

Returns the X coordinate of a point geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double). The X axis is determined from its CRS definition.

In [40]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?x
WHERE {
  my:FExactGeom geo:asWKT ?literal .
  BIND(geof:X(?literal) AS ?x)
}
"""),["literal"],None)

Variable,Value
literal,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
x,-83.4


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:Y Function

Returns the Y coordinate of a Point geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double). The Y axis is determined by its CRS definition.

In [41]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?y
WHERE {
  my:FExactGeom geo:asWKT ?literal .
  BIND(geof:Y(?literal) AS ?y)
}
"""),["literal"],None)

Variable,Value
literal,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
y,34.4


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:Z Function

Returns the Z coordinate of a Point geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double). The Z axis is determined by its CRS definition.

In [42]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?z
WHERE {
  my:NPointGeom geo:asWKT ?literal .
  BIND(geof:Z(?literal) AS ?z)
}
"""),["literal"],None)

Variable,Value
literal,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point Z (-77.059 38.9 1)
z,1.0


Map(center=[38.9, -77.059], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

## Geometry Measurement Functions

These functions perform measurements on geometries and return either a numeric result or a geometry derived from the respective measurement

### geofext:azimuth Function

Calculates the azimuth of a given geometry and returns the angle as a [xsd:double](http://www.w3.org/2001/XMLSchema#double) 

In [43]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?azimuth
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:azimuth(?aLiteral) as ?azimuth)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
azimuth,90.0


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:closestPoint Function

Returns the closest point between two input geometries. This point is the first point of the shortest line between the geometries.

In [44]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?cPoint
WHERE {
  my:A geo:hasGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:closestPoint(?aLiteral, ?dLiteral) as ?cPoint)
}
"""),["aLiteral","dLiteral","cPoint"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
cPoint,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT (-83.2 34.1)


Map(center=[34.1, -83.2], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:farthestCoordinate Function

In [45]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?farthestCoord
WHERE {
  my:A my:hasPointGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:farthestCoordinate(?aLiteral, ?dLiteral) as ?farthestCoord)
}
"""),["aLiteral","dLiteral","farthestCoord"],None)

Variable,Value
aLiteral,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.3)
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
farthestCoord,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT (-83.1 34)


Map(center=[34.0, -83.1], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:frechetDistance Function

Calculates the FrechetDistance between two input geometries and returns the result as [xsd:double](http://www.w3.org/2001/XMLSchema#double)

In [46]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?fdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:frechetDistance(?cLiteral, ?fLiteral) as ?fdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
fdistance,0.41231056256177195


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:hausdorffDistance Function

Calculates the haussdorff distances between two geometries and returns the result as [xsd:double](http://www.w3.org/2001/XMLSchema#double)

In [47]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?hdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:hausdorffDistance(?cLiteral, ?fLiteral) as ?hdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
hdistance,0.41231056256177195


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:interpolatePoint Function

In [48]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?eLiteral ?interpolated
WHERE {
  my:E geo:hasDefaultGeometry ?eGeom .
  ?eGeom geo:asWKT ?eLiteral .
  BIND (geof:interpolatePoint(?eLiteral, "2"^^xsd:double) as ?interpolated)
}
"""),["eLiteral","interpolated"],None)

LINESTRING (-83.4 34, -83.3 34.3)
2.0


Variable,Value
eLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LineString(-83.4 34.0, -83.3 34.3)"
interpolated,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT (-83.3 34.3)


Map(center=[34.3, -83.3], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:longestLine Function 🧊

Returns the longest line between two geometries in the literal format and CRS of the first input geometry.

In [49]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT  ?aLiteral ?dLiteral ?longestLine
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:longestLine(?aLiteral, ?dLiteral) as ?longestLine)
}
"""),["aLiteral","dLiteral","longestLine"],None)

LONGESTLINE


Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
longestLine,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LINESTRING (-83.6 34.5, -83.1 34)"


Map(center=[34.25, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:maxDistance Function 🧊

Calculates the largest distance between two geometries in the unit of the CRS of the first geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double)

In [50]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT  ?aLiteral ?dLiteral ?maxDistance
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:D geo:hasDefaultGeometry ?dGeom .
  ?dGeom geo:asWKT ?dLiteral .
  BIND (geof:maxDistance(?aLiteral, ?dLiteral) as ?maxDistance)
}
"""),["aLiteral","dLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
maxDistance,0.7071067811865476


Map(center=[34.099999999999994, -83.19999999999997], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:maximumInscribedCircle Function

In [51]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?maxicirc
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:maximumInscribedCircle(?aLiteral) as ?maxicirc)
}
"""),["aLiteral","maxicirc"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
maxicirc,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.20000000000002 34.3, -83.20096305466558 34.280396571934084, -83.20384294391937 34.260981935596774, -83.20861193285357 34.241943064549105, -83.21522409349775 34.223463313526985, -83.22361574713034 34.2057206526348, -83.23370607753951 34.18888595339608, -83.24539790932747 34.173121343167274, -83.2585786437627 34.158578643762695, -83.27312134316729 34.14539790932746, -83.28888595339609 34.1337060775395, -83.30572065263482 34.12361574713034, -83.32346331352699 34.11522409349775, -83.34194306454911 34.10861193285356, -83.36098193559678 34.10384294391936, -83.3803965719341 34.10096305466557, -83.4 34.10000000000001, -83.41960342806591 34.10096305466557, -83.43901806440323 34.10384294391936, -83.4580569354509 34.10861193285356, -83.47653668647303 34.11522409349775, -83.4942793473652 34.12361574713034, -83.51111404660392 34.1337060775395, -83.52687865683272 34.14539790932746, -83.54142135623731 34.158578643762695, -83.55460209067255 34.173121343167274, -83.5662939224605 34.18888595339608, -83.57638425286967 34.2057206526348, -83.58477590650226 34.223463313526985, -83.59138806714644 34.241943064549105, -83.59615705608064 34.260981935596774, -83.59903694533443 34.280396571934084, -83.6 34.3, -83.59903694533443 34.31960342806591, -83.59615705608064 34.33901806440322, -83.59138806714644 34.35805693545089, -83.58477590650226 34.37653668647301, -83.57638425286967 34.39427934736519, -83.5662939224605 34.41111404660391, -83.55460209067255 34.42687865683272, -83.54142135623731 34.4414213562373, -83.52687865683272 34.45460209067254, -83.51111404660392 34.466293922460494, -83.4942793473652 34.47638425286966, -83.47653668647303 34.48477590650224, -83.4580569354509 34.49138806714643, -83.43901806440323 34.496157056080634, -83.41960342806591 34.499036945334424, -83.4 34.499999999999986, -83.3803965719341 34.499036945334424, -83.36098193559678 34.496157056080634, -83.34194306454911 34.49138806714643, -83.32346331352699 34.48477590650224, -83.30572065263482 34.47638425286966, -83.28888595339609 34.466293922460494, -83.27312134316729 34.45460209067254, -83.2585786437627 34.4414213562373, -83.24539790932747 34.42687865683272, -83.23370607753951 34.41111404660391, -83.22361574713034 34.39427934736519, -83.21522409349775 34.37653668647301, -83.20861193285357 34.35805693545089, -83.20384294391937 34.33901806440322, -83.20096305466558 34.31960342806591, -83.20000000000002 34.3))"


Map(center=[34.30000000000001, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### geofext:minimumBoundingRadius Function

Calculates the radius of a minimum bounding circle as [xsd:double](http://www.w3.org/2001/XMLSchema#double)

In [52]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bradius
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minimumBoundingRadius(?aLiteral) as ?bradius)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bradius,0.28284271247460796


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:minimumClearance Function

Calculates the minimum clearance of a given input geometry and returns the result as [xsd:double](http://www.w3.org/2001/XMLSchema#double)

In [53]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?mclearance
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minimumClearance(?aLiteral) as ?mclearance)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
mclearance,0.3999999999999915


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:minimumClearanceLine Function

In [54]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?mclearancel
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minimumClearanceLine(?aLiteral) as ?mclearancel)
}
"""),["aLiteral","mclearancel"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
mclearancel,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LINESTRING (-83.6 34.1, -83.2 34.1)"


Map(center=[34.1, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:pointOnSurface Function

Returns a point on the surface of a polygon geometry.

In [55]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?pointOnSurface
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:pointOnSurface(?literal) AS ?pointOnSurface)
}
"""),["literal","pointOnSurface"],None)

POINT (-83.4 34.3)


Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
pointOnSurface,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT (-83.4 34.3)


Map(center=[34.3, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:shortestLine Function 🧊

Calculates the shortest line between two given geometries in the CRS of the first input geometry.

In [56]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?literal2 ?sline
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  my:OExactGeom geo:asWKT ?literal2 .
  BIND(geof:shortestLine(?literal,?literal2) AS ?sline)
}
"""),["literal","literal2","sline"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
literal2,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
sline,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LINESTRING (-83.2 34.5, 2.39251 47.44793)"


Map(center=[40.973965, -40.403745], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Geometry Modification Functions

Functions which modify geometries by adding, editing or removing coordinates or transforming the geometry without changing its CRS

### geofext:appendPoint Function

Appends a point to the input geometry. Returns the modified geometry in the literal format and CRS of the input geometry.

In [57]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?appended
WHERE {
  BIND("POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"^^geo:wktLiteral AS ?literal)
  BIND(geof:appendPoint(?literal,"POINT(-84.2 33.5)"^^geo:wktLiteral) AS ?appended)
}
"""),["literal","appended"],None)

Variable,Value
literal,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"
appended,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -84.2 33.5, -83.6 34.1))"


Map(center=[34.128571428571426, -83.5142857142857], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### geofext:addPoint Function

Add a point at the given index to the input geometry. Returns the modified geometry in the literal format and CRS of the input geometry.

In [58]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?added
WHERE {
  BIND("POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"^^geo:wktLiteral AS ?literal)
  BIND(geof:addPoint(?literal,"POINT(-84.2 33.5)"^^geo:wktLiteral,2) AS ?added)
}
"""),["literal","added"],None)

Variable,Value
literal,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"
added,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.6 34.5, -84.2 33.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"


Map(center=[33.63333333333334, -84.7333333333333], controls=(ZoomControl(options=['position', 'zoom_in_text', …

### geofext:clipByRect Function

In [59]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?dliteral ?clipped
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  my:DExactGeom geo:asWKT ?dliteral .
  BIND(geof:clipByRect(?literal,?dliteral) AS ?clipped)
}
"""),["literal","dliteral","clipped"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dliteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
clipped,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.3 34.1, -83.3 34.2, -83.2 34.2, -83.2 34.1, -83.3 34.1))"


Map(center=[34.15, -83.25], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:force2D Function

Creates a 2D geometry from a given geometry of a higher dimension by reducing its dimensionality

In [60]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?force2D
WHERE {
  my:NExactGeom geo:asWKT ?literal .
  BIND(geof:force2D(?literal) AS ?force2D)
}
"""),["literal","force2D"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon Z((-77.089005 38.913574 1,-77.029953 38.913574 2,-77.029953 38.886321 2,-77.089005 38.886321 1,-77.089005 38.913574 1))"
force2D,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-77.089005 38.913574, -77.029953 38.913574, -77.029953 38.886321, -77.089005 38.886321, -77.089005 38.913574))"


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### geofext:force3D Function

Creates a 3D geometry by adding a given Z coordinate to every point of the input geometry

In [61]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?force3D
WHERE {
  my:NExactGeom geo:asWKT ?literal .
  BIND(geof:force3D(?literal,"5.0"^^xsd:double) AS ?force3D)
}
"""),["literal","force3D"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon Z((-77.089005 38.913574 1,-77.029953 38.913574 2,-77.029953 38.886321 2,-77.089005 38.886321 1,-77.089005 38.913574 1))"


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### geofext:reducePrecision Function

In [62]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?reduced
WHERE {
  BIND("POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"^^geo:wktLiteral AS ?literal)
  BIND(geof:reducePrecision(?literal,1.0) AS ?reduced)
}
"""),["literal","reduced"],None)

Variable,Value
literal,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"
reduced,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-84 35, -83 35, -83 34, -84 34, -84 35))"


Map(center=[34.5, -83.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:removePoint Function

Removes a point from a given geometry at the given index and returns the new geometry in the literal format and CRS of the input geometry.

In [63]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?removed
WHERE {
  BIND("POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"^^geo:wktLiteral AS ?literal)
  BIND(geof:removePoint(?literal,3) AS ?removed)
}
"""),["literal","removed"],None)

Variable,Value
literal,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"
removed,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.1, -83.6 34.1))"


Map(center=[34.23333333333333, -83.46666666666665], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### geofext:removeRepeatedPoints Function

In [64]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal  ?rep
WHERE {
  BIND("POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"^^geo:wktLiteral AS ?literal)
  BIND(geof:removeRepeatedPoints(?literal) AS ?rep)
}
"""),["literal"],None)

Variable,Value
literal,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"
rep,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:reverse Function

In [65]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?reverse
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:reverse(?literal) AS ?reverse)
}
"""),["literal","reverse"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
reverse,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:setPoint Function

Sets the point at the given index of the input geometry if it exists. Returns the modified geometry in the literal format and CRS of the input geometry.

In [66]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?changed
WHERE {
  BIND("POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"^^geo:wktLiteral AS ?literal)
  BIND(geof:setPoint(?literal,"POINT(-84.2 33.5)"^^geo:wktLiteral,3) AS ?changed)
}
"""),["literal","changed"],None)

Variable,Value
literal,"POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.6 34.1))"
changed,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.6 34.5, -83.6 34.5, -84.2 33.5, -83.2 34.1, -83.6 34.1))"


Map(center=[33.96666666666667, -83.73333333333333], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### geofext:simplify Function

In [67]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?simple
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:simplify(?literal,"1.5"^^xsd:double) AS ?simple)
}
"""),["literal","simple"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
simple,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.5, -83.2 34.1, -83.2 34.5, -83.6 34.5))"


Map(center=[34.36666666666667, -83.33333333333333], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### geofext:smooth Function

Calculates a smoothed version of the given geometry using the Chaikin Smooting algorithm in the same literal format and CRS than the input geometry

In [68]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?smoothed
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:smooth(?literal,0.1) AS ?smoothed)
}
"""),["literal","smoothed"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
smoothed,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.40625 34.1, -83.39375 34.1, -83.381640625 34.100390625, -83.36992187500002 34.101171875000006, -83.35859375 34.10234375, -83.34765625 34.10390625, -83.337109375 34.105859375, -83.32695312499999 34.108203125, -83.31718750000002 34.110937500000006, -83.3078125 34.1140625, -83.298828125 34.117578125, -83.290234375 34.121484375, -83.28203125 34.12578125, -83.27421875 34.130468750000006, -83.266796875 34.135546875, -83.259765625 34.141015625, -83.25312500000001 34.146875, -83.246875 34.153125, -83.24101562500002 34.159765625000006, -83.235546875 34.166796875, -83.23046875 34.17421875, -83.22578125 34.182031249999994, -83.221484375 34.190234375, -83.217578125 34.198828125000006, -83.2140625 34.2078125, -83.2109375 34.2171875, -83.208203125 34.226953124999994, -83.205859375 34.237109375, -83.20390625 34.247656250000006, -83.20234375000001 34.25859375, -83.201171875 34.269921875, -83.200390625 34.281640624999994, -83.2 34.29375, -83.2 34.306250000000006, -83.20039062500001 34.318359375, -83.20117187500001 34.330078125, -83.20234375000001 34.34140625, -83.20390625 34.35234375, -83.205859375 34.362890625, -83.20820312500001 34.373046875, -83.21093750000001 34.3828125, -83.21406250000001 34.3921875, -83.217578125 34.401171875, -83.221484375 34.409765625, -83.22578125000001 34.41796875, -83.23046875000001 34.42578125, -83.23554687500001 34.433203125, -83.241015625 34.440234375, -83.246875 34.446875, -83.25312500000001 34.453125, -83.25976562500001 34.458984375, -83.26679687500001 34.464453125, -83.27421875 34.46953125, -83.28203125 34.47421875, -83.29023437500001 34.478515625, -83.298828125 34.482421875, -83.3078125 34.4859375, -83.3171875 34.4890625, -83.32695312499999 34.491796875, -83.337109375 34.494140625, -83.34765625 34.49609375, -83.35859375 34.49765625, -83.369921875 34.498828125, -83.38164062499999 34.499609375, -83.39375 34.5, -83.40625 34.5, -83.418359375 34.499609375, -83.43007812500001 34.498828125, -83.44140625 34.49765625, -83.45234375 34.49609375, -83.462890625 34.494140625, -83.473046875 34.491796875, -83.48281250000001 34.4890625, -83.4921875 34.4859375, -83.501171875 34.482421875, -83.509765625 34.478515625, -83.51796875 34.47421875, -83.52578125000001 34.46953125, -83.533203125 34.464453125, -83.540234375 34.458984375, -83.546875 34.453125, -83.553125 34.446875, -83.55898437500001 34.440234375, -83.564453125 34.433203125, -83.56953125 34.42578125, -83.57421875 34.41796875, -83.578515625 34.409765625, -83.582421875 34.401171875, -83.5859375 34.3921875, -83.5890625 34.3828125, -83.59179687499999 34.373046875, -83.594140625 34.362890625, -83.59609375 34.35234375, -83.59765625 34.34140625, -83.598828125 34.330078125, -83.599609375 34.318359375, -83.6 34.306250000000006, -83.6 34.29375, -83.59960937499999 34.281640625, -83.59882812500001 34.26992187500001, -83.59765624999999 34.258593749999996, -83.59609375 34.247656250000006, -83.594140625 34.237109375, -83.59179687499999 34.226953125, -83.58906250000001 34.21718750000001, -83.58593749999999 34.207812499999996, -83.582421875 34.198828125000006, -83.578515625 34.190234375, -83.57421874999999 34.18203125, -83.56953125 34.17421875, -83.56445312499999 34.166796875, -83.558984375 34.159765625000006, -83.553125 34.153125, -83.54687499999999 34.146875, -83.54023437500001 34.141015625, -83.533203125 34.135546875, -83.52578125 34.130468750000006, -83.51796875 34.12578125, -83.50976562499999 34.121484375, -83.501171875 34.117578125, -83.4921875 34.1140625, -83.4828125 34.110937500000006, -83.473046875 34.108203125, -83.462890625 34.105859375, -83.45234375 34.10390625, -83.44140625 34.10234375, -83.430078125 34.101171875000006, -83.418359375 34.100390625, -83.40625 34.1))"


Map(center=[34.3, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

## Geometry Relations

Functions which calculate relations between geometries

### geofext:equalsExact Function

In [69]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bLiteral (xsd:boolean(?sfEquals) as ?AequalsA) (xsd:boolean(?sfEquals2) as ?AequalsB)
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  my:B geo:hasDefaultGeometry ?bGeom .
  ?bGeom geo:asWKT ?bLiteral .
  BIND (geof:equalsExact(?aLiteral, ?aLiteral) as ?sfEquals)
  BIND (geof:equalsExact(?aLiteral, geof:reverse(?aLiteral)) as ?sfEquals2)
}
"""),["aLiteral","bLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.4 34.1, -83.4 34.3, -83.6 34.3, -83.6 34.1))"
AequalsA,true
AequalsB,false


Map(center=[34.2, -83.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:fullyWithinDistance Function

Calculates whether the first geometry is fully within a given distance to a second geometry and returns the result as [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [70]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?fwdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:fullyWithinDistance(?cLiteral, ?fLiteral,"5.0"^^xsd:double) as ?fwdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
fwdistance,True


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:metricWithinDistance Function

In [71]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral (xsd:boolean(?wdistance) as ?wddistance)
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:metricWithinDistance(?cLiteral, ?fLiteral,"5.0"^^xsd:double) as ?wdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
wddistance,false


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:pointInsideCircle Function

In [72]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?pointINsideCircle
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  my:APointGeom geo:asWKT ?pliteral .
  BIND(geof:pointInsideCircle(?literal,?pliteral,10.0) AS ?pointInsideCircle)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:sharedPaths Function

In [73]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?literal2 ?spaths
WHERE {
  my:EExactGeom geo:asWKT ?literal .
  my:EExactGeom geo:asWKT ?literal2 .
  BIND(geof:sharedPaths(?literal,?literal2) AS ?spaths)
}
"""),["literal","literal2","spaths"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LineString(-83.4 34.0, -83.3 34.3)"
literal2,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LineString(-83.4 34.0, -83.3 34.3)"
spaths,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> GEOMETRYCOLLECTION (MULTILINESTRING ((-83.4 34, -83.3 34.3)), MULTILINESTRING EMPTY)"


Map(center=[34.15, -83.35], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:withinDistance Function

Checks whether the first geometry is within a given distance to the second geometry. The result is given as [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [74]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?wdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:withinDistance(?cLiteral, ?fLiteral,"5.0"^^xsd:double) as ?wdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

## Geometry Transformations

Transformations which take a geometry input and moves, scales, translates or otherwise transforms it within the same or into a different coordinate reference system.

### geofext:affineTransformation

Calculates and affine transformation either in a 2D or in a 3D space. It expects either a [a, b, d, e, xoff, yoff] matrix for 2D conversions or a [a, b, c, d, e, f, g, h, i, xoff, yoff, zoff] matrix for 3D conversion

In [75]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?transformed
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:affineTransformation(?literal,"[1.45 1.0 0.0 1.0 2.0 -1.0]"^^geo:matrixLiteral) AS ?transformed)
}
"""),["literal","transformed"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
transformed,"POLYGON ((-85.11999999999998 33.1, -84.53999999999999 33.1, -84.14 33.5, -84.71999999999998 33.5, -85.11999999999998 33.1))"


Map(center=[33.3, -84.62999999999998], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:constrainedDelaunay Function

In [76]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?delau
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:constrainedDelaunay(?literal) AS ?delau)
}
"""),["literal","delau"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
delau,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> GEOMETRYCOLLECTION (POLYGON ((-83.6 34.1, -83.6 34.5, -83.2 34.5, -83.6 34.1)), POLYGON ((-83.2 34.5, -83.2 34.1, -83.6 34.1, -83.2 34.5)))"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:delaunayTriangles Function

In [77]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?delau
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:delaunayTriangles(?literal) AS ?delau)
}
"""),["literal","delau"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
delau,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> GEOMETRYCOLLECTION (POLYGON ((-83.6 34.5, -83.6 34.1, -83.2 34.1, -83.6 34.5)), POLYGON ((-83.6 34.5, -83.2 34.1, -83.2 34.5, -83.6 34.5)))"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

### geofext:lineMerge Function

In [78]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?mlstring ?merged
WHERE {
  BIND("MULTILINESTRING((0 0, 1 0),(1 0, 2 0))"^^geo:wktLiteral AS ?mlstring)
  BIND(geof:lineMerge(?mlstring) AS ?merged)
}
"""),["mlstring","merged"],None)

Variable,Value
mlstring,"MULTILINESTRING((0 0, 1 0),(1 0, 2 0))"
merged,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LINESTRING (0 0, 1 0, 2 0)"


Map(center=[0.0, 1.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


### geofext:makeValid Function

In [79]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?invalidLiteral ?valid
WHERE {
  BIND("LINESTRING(0 0, 0 0)"^^geo:wktLiteral AS ?invalidLiteral)
  BIND(geof:makeValid(?invalidLiteral) AS ?valid)
}
"""),["invalidLiteral","valid"],None)

Variable,Value
invalidLiteral,"LINESTRING(0 0, 0 0)"
valid,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT (0 0)


Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

### geofext:offsetCurve Function

In [80]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?offsetCurve
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:offsetCurve(?literal,1) AS ?offsetCurve)
}
"""),["literal","offsetCurve"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
offsetCurve,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> LINESTRING (-84.6 34.1, -84.6 34.5, -84.58078528040322 34.69509032201613, -84.52387953251129 34.88268343236509, -84.43146961230254 35.0555702330196, -84.30710678118655 35.207106781186546, -84.1555702330196 35.33146961230255, -83.98268343236508 35.423879532511286, -83.79509032201612 35.48078528040323, -83.6 35.5, -83.2 35.5, -83.00490967798387 35.48078528040323, -82.81731656763492 35.423879532511286, -82.6444297669804 35.33146961230255, -82.49289321881345 35.207106781186546, -82.36853038769746 35.0555702330196, -82.27612046748871 34.88268343236509, -82.21921471959678 34.69509032201613, -82.2 34.5, -82.2 34.1, -82.21921471959678 33.90490967798387, -82.27612046748871 33.71731656763491, -82.36853038769746 33.5444297669804, -82.49289321881345 33.392893218813455, -82.6444297669804 33.26853038769745, -82.81731656763492 33.176120467488715, -83.00490967798387 33.11921471959677, -83.2 33.1, -83.6 33.1, -83.79509032201612 33.11921471959677, -83.98268343236508 33.176120467488715, -84.1555702330196 33.26853038769745, -84.30710678118655 33.392893218813455, -84.43146961230254 33.5444297669804, -84.52387953251129 33.71731656763491, -84.58078528040322 33.90490967798387, -84.6 34.1)"


Map(center=[34.300000000000004, -83.40000000000002], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:scale Function

Returns a scaled version of a geometry in the literal format and CRS of the input geometry by applying three scaling factors for the X,Y and Z axis respectively.
If a Z axis is not present, the Z factor will be ignored.


In [81]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?scale
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:scale(?literal,"2.0"^^xsd:double,"2.0"^^xsd:double,"2.0"^^xsd:double) AS ?scale)
}
"""),["literal","scale"],None)

POLYGON ((-83.79999999999998 33.900000000000006, -83 33.900000000000006, -83 34.7, -83.79999999999998 34.7, -83.79999999999998 33.900000000000006))


Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
scale,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.79999999999998 33.900000000000006, -83 33.900000000000006, -83 34.7, -83.79999999999998 34.7, -83.79999999999998 33.900000000000006))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:snap Function

In [82]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dLiteral ?snapped
WHERE {
  my:AExactGeom geo:asWKT ?aLiteral .
  my:DExactGeom geo:asWKT ?dLiteral .
  BIND(geof:snap(?aLiteral,?dLiteral) AS ?snapped)
}
"""),["aLiteral","dLiteral","snapped"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
dLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.3 34.0, -83.1 34.0, -83.1 34.2, -83.3 34.2, -83.3 34.0))"
snapped,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:split Function

In [83]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?eLiteral ?splitted
WHERE {
  my:AExactGeom geo:asWKT ?aLiteral .
  BIND("LineString(-83.4 34.0, -83.3 34.3, -85.0 35.0)"^^geo:wktLiteral AS ?eLiteral)
  BIND(geof:split(?aLiteral,?eLiteral) AS ?splitted)
}
"""),["aLiteral","eLiteral","splitted"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
eLiteral,"LineString(-83.4 34.0, -83.3 34.3, -85.0 35.0)"
splitted,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> GEOMETRYCOLLECTION (POLYGON ((-83.36666666666667 34.1, -83.6 34.1, -83.6 34.423529411764704, -83.3 34.3, -83.36666666666667 34.1)), POLYGON ((-83.6 34.423529411764704, -83.6 34.5, -83.2 34.5, -83.2 34.1, -83.36666666666667 34.1, -83.3 34.3, -83.6 34.423529411764704)))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:translate Function

In [84]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?translate
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:translate(?literal,"1.0"^^xsd:double,"1.0"^^xsd:double,"1.0"^^xsd:double) AS ?translate)
}
"""),["literal","translate"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
translate,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-82.6 35.1, -82.2 35.1, -82.2 35.5, -82.6 35.5, -82.6 35.1))"


Map(center=[35.300000000000004, -82.40000000000002], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### geofext:transformCRS84 Function

In [85]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?transformed
WHERE {
  my:LExactGeom geo:asWKT ?literal .
  BIND(geof:transformCRS84(?literal) AS ?transformed)
}
"""),["literal","transformed"],None)

Variable,Value
literal,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-88.38 31.95)
transformed,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT (-88.38 31.95)


Map(center=[31.95, -88.38], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

### geofext:voronoiLines Function

In [86]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?voro
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:voronoiLines(?literal) AS ?voro)
}
"""),["literal","voro"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
voro,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> MULTILINESTRING ((-83.4 34.3, -83.4 34.9), (-82.80000000000001 34.3, -83.4 34.3), (-83.4 34.3, -84 34.3), (-83.4 34.3, -83.4 33.7))"


Map(center=[34.3, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### geofext:voronoiPolygons Function

In [87]:
processQueryResults(g.query("""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?voro
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:voronoiPolygons(?literal) AS ?voro)
}
"""),["literal","voro"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
voro,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> GEOMETRYCOLLECTION (POLYGON ((-82.80000000000001 34.9, -82.80000000000001 34.3, -83.4 34.3, -83.4 34.9, -82.80000000000001 34.9)), POLYGON ((-84 34.9, -83.4 34.9, -83.4 34.3, -84 34.3, -84 34.9)), POLYGON ((-84 33.7, -84 34.3, -83.4 34.3, -83.4 33.7, -84 33.7)), POLYGON ((-82.80000000000001 33.7, -83.4 33.7, -83.4 34.3, -82.80000000000001 34.3, -82.80000000000001 33.7)))"


Map(center=[34.3, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…